# Model V6 - Hospital Density Moderating Seasonal Amplitude

Model V6 extends V3 by replacing flat priors on the seasonal coefficients with hospital-density-informed priors.

$$\beta_i \sim \mathcal{N}(\mu_\beta + \eta_\beta h_i,\; \tau_\beta)$$
$$\gamma_i \sim \mathcal{N}(\mu_\gamma + \eta_\gamma h_i,\; \tau_\gamma)$$

This notebook reconstructs posterior fitted values, checks residual diagnostics, and tests whether higher ED hospital density is associated with weaker seasonal swings in trolley rates.

In [ ]:
from model_helpers import *
import os

sns.set_theme(style='whitegrid', font_scale=1.15)

VERSION = 'v6'
DATA_DIR = f'../../data/models/{VERSION}/'
PLOT_DIR = DATA_DIR + 'plots/'
os.makedirs(PLOT_DIR, exist_ok=True)

df_og, raw_df, regions, n_region = load_model_data(VERSION)
hosp_df = load_region_covariate('../../data/hospitals_per_region.csv', regions, 'EDHospitalsPer10k', center=True)
hosp_df

## Load model parameters and reconstruct fitted values

In [ ]:
n_weeks = df_og.shape[1] - 1
ev = build_event_indicators(n_weeks, regions)
df_mu, df_mu_lower, df_mu_upper, phi_mean = reconstruct_mu(raw_df, regions, n_weeks, ev)

df_og = transpose_observed(df_og)
df_ar1 = compute_ar1_fitted(df_mu, df_og, phi_mean)
df_residuals, df_std_resid = compute_residuals(df_og, df_ar1)

df_fitted = df_ar1.copy()
df_fitted['time'] = np.arange(1, len(df_fitted) + 1)
df_fitted = df_fitted.melt(id_vars='time', var_name='Region', value_name='fitted')
df_fitted.to_csv(DATA_DIR + 'fitted.csv', index=False)

df_residuals_long = df_residuals.copy()
df_residuals_long['time'] = np.arange(1, len(df_residuals_long) + 1)
df_residuals_long = df_residuals_long.melt(id_vars='time', var_name='Region', value_name='residual')
df_residuals_long.to_csv(DATA_DIR + 'residuals.csv', index=False)

df_ar1.head()

## Hospital density by region

In [ ]:
hosp_summary = hosp_df.rename(columns={
    'EDHospitalsPer10k': 'ED hospitals per 10k',
    'EDHospitalsPer10k_centered': 'Centered density'
})
hosp_summary.sort_values('ED hospitals per 10k', ascending=False).round(4)

## Posterior mean of the deterministic mean function

In [ ]:
plot_mu(df_mu, df_mu_lower, df_mu_upper, PLOT_DIR + 'mu_fit.png')

## Full AR(1) fitted values and residual diagnostics

In [ ]:
plot_ar1_fit(df_ar1, PLOT_DIR + 'ar1_fit.png')
plot_residuals_ts(df_std_resid, PLOT_DIR + 'residuals_ts.png')
plot_residuals_acf(df_std_resid, PLOT_DIR + 'residuals_acf.png')
plot_residuals_vs_fitted(df_std_resid, df_ar1, PLOT_DIR + 'residuals_vs_fitted.png')
plot_residuals_qq(df_std_resid, PLOT_DIR + 'residuals_qq.png')

## Global seasonal moderation parameters

In [ ]:
global_params = summarize_global_parameters(
    raw_df,
    ['mu_beta', 'mu_gamma', 'eta_beta', 'eta_gamma', 'tau_beta', 'tau_gamma', 'phi']
)
global_params.to_csv(DATA_DIR + 'global_parameters.csv', index=False)
global_params

In [ ]:
rows = []
for param in ['eta_beta', 'eta_gamma']:
    vals = raw_df[param].values
    rows.append({
        'Parameter': param,
        'Mean': vals.mean(),
        'SD': vals.std(),
        '2.5%': np.quantile(vals, 0.025),
        '97.5%': np.quantile(vals, 0.975),
        'P(>0)': (vals > 0).mean(),
        'P(<0)': (vals < 0).mean(),
    })

df_eta = pd.DataFrame(rows).round(4)
df_eta.to_csv(DATA_DIR + 'eta_summary.csv', index=False)
df_eta

## Annual cycle amplitude

$$\hat{A}_i = \sqrt{\beta_i^2 + \gamma_i^2}$$

The main V6 question is whether regions with higher ED hospital density tend to have smaller posterior seasonal amplitudes.

In [ ]:
epsilon = 0.5
ampl = compute_amplitude(raw_df, regions)

results = []
for region in regions:
    a = ampl[region]
    density = hosp_df.loc[hosp_df['Region'] == region, 'EDHospitalsPer10k'].iat[0]
    results.append({
        'Region': region,
        'EDHospitalsPer10k': density,
        'Mean A': a.mean(),
        'Median A': np.median(a),
        '2.5%': np.quantile(a, 0.025),
        '97.5%': np.quantile(a, 0.975),
        'P(A > 0.5)': (a > epsilon).mean(),
    })

df_ampl_overall = pd.DataFrame(results).sort_values('Mean A').round(4)
df_ampl_overall.to_csv(DATA_DIR + 'amplitude_overall.csv', index=False)
df_ampl_overall

In [ ]:
epsilons = np.linspace(0, 3, 200)

fig, ax = plt.subplots(figsize=(8, 4), dpi=150, layout='constrained')
for region in regions:
    probs = [(ampl[region] > e).mean() for e in epsilons]
    ax.plot(epsilons, probs, label=region)

ax.axvline(epsilon, linestyle='--', color='grey', linewidth=1, label=r'$\varepsilon = 0.5$')
ax.set_xlabel(r'Threshold $\varepsilon$ (trolleys per 10k)')
ax.set_ylabel(r'$P(A_i > \varepsilon)$')
ax.set_title('Amplitude Sensitivity: Posterior Exceedance Probability')
ax.legend(fontsize=7, loc='upper right')
ax.set_ylim(0, 1.02)
sns.despine()
fig.savefig(PLOT_DIR + 'amplitude_overall.png', bbox_inches='tight', dpi=150)
plt.show()

df_ampl_pw = pairwise_test(ampl, regions, epsilon=epsilon)
df_ampl_pw.to_csv(DATA_DIR + 'amplitude_pairwise.csv', index=False)
pairwise_heatmap(
    df_ampl_pw,
    'P(diff > 0)',
    'Amplitude: P(Region A > Region B)',
    PLOT_DIR + 'amplitude_pairwise.png'
)
df_ampl_pw

## Phase cycles — overall test

In [ ]:
ci_lower_q, ci_upper_q = 0.025, 0.975

phase_rows = []
for i, region in enumerate(regions):
    b = raw_df[f'beta[{i+1}]'].values
    g = raw_df[f'gamma[{i+1}]'].values
    phase_rad = np.arctan2(g, b)
    lo_rad, hi_rad = np.quantile(phase_rad, [ci_lower_q, ci_upper_q])
    mean_wk = ((52 / (2 * np.pi)) * phase_rad.mean()) % 52
    lo_wk = ((52 / (2 * np.pi)) * lo_rad) % 52
    hi_wk = ((52 / (2 * np.pi)) * hi_rad) % 52
    ci_width = (hi_wk - lo_wk) % 52
    phase_rows.append({
        'Region': region,
        'Mean Peak Week': mean_wk,
        '2.5%': lo_wk,
        '97.5%': hi_wk,
        'CI Width': ci_width,
        'Identifiable': 'No' if ci_width > 40 else 'Yes',
    })

df_phase_overall = pd.DataFrame(phase_rows).sort_values('Mean Peak Week').round(3)
df_phase_overall.to_csv(DATA_DIR + 'phase_overall.csv', index=False)

df_plot = df_phase_overall.sort_values('Mean Peak Week')
lo_wk = df_plot['2.5%'].values
hi_wk = df_plot['97.5%'].values

def ci_width(lo, hi):
    return (hi - lo) % 52

fig, ax = plt.subplots(figsize=(9, 4), dpi=150, layout='constrained')
y_pos = list(range(len(df_plot)))
ax.axvspan(48, 52, alpha=0.12, color='steelblue', label='Winter (wk 48-4)')
ax.axvspan(0, 4, alpha=0.12, color='steelblue')

for y, lo, hi, region in zip(y_pos, lo_wk, hi_wk, df_plot['Region'].values):
    width = ci_width(lo, hi)
    if width > 40:
        ax.hlines(y, 0, 52, color='grey', linewidth=1.5, linestyle='--', zorder=1)
        ax.annotate('(no clear cycle)', xy=(52, y), fontsize=7, color='grey', va='center', ha='right')
    elif lo <= hi:
        ax.hlines(y, lo, hi, color='grey', linewidth=2.5, zorder=2)
    else:
        ax.hlines(y, lo, 52, color='grey', linewidth=2.5, zorder=2)
        ax.hlines(y, 0, hi, color='grey', linewidth=2.5, zorder=2)

ax.scatter(df_plot['Mean Peak Week'].values, y_pos, s=80, zorder=3, color='#2c7bb6', edgecolors='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels([SHORT_NAMES.get(r, r) for r in df_plot['Region'].values])
ax.set_xlabel('Peak week of year')
ax.set_xlim(0, 52)
ax.set_ylim(-0.5, len(y_pos) - 0.5)
ax.set_title('Seasonal Peak Timing by Region')
ax.legend(loc='upper right', fontsize=8)
sns.despine(left=True)
fig.savefig(PLOT_DIR + 'seasonal_phase_test.png', bbox_inches='tight', dpi=150)
plt.show()

df_phase_overall

In [ ]:
n_winter_weeks = 9
n_summer_weeks = 9
prior_winter = n_winter_weeks / 52
prior_summer = n_summer_weeks / 52

rows = []
for i, region in enumerate(regions):
    b = raw_df[f'beta[{i+1}]'].values
    g = raw_df[f'gamma[{i+1}]'].values
    phase_rad = np.arctan2(g, b)
    peak_weeks = (52 / (2 * np.pi) * phase_rad) % 52
    mean_peak_wk = ((52 / (2 * np.pi)) * phase_rad.mean()) % 52
    prob_winter = ((peak_weeks >= 48) | (peak_weeks <= 4)).mean()
    prob_summer = ((peak_weeks >= 22) & (peak_weeks <= 30)).mean()
    post_odds_w = prob_winter / max(1 - prob_winter, 1e-10)
    prior_odds_w = prior_winter / (1 - prior_winter)
    bf_winter = post_odds_w / prior_odds_w
    post_odds_s = prob_summer / max(1 - prob_summer, 1e-10)
    prior_odds_s = prior_summer / (1 - prior_summer)
    bf_summer = post_odds_s / prior_odds_s
    rows.append({
        'Region': region,
        'Mean peak week': mean_peak_wk,
        'P(winter)': prob_winter,
        'BF(winter)': bf_winter,
        'P(summer)': prob_summer,
        'BF(summer)': bf_summer,
    })

df_season_phase_test = pd.DataFrame(rows).sort_values('P(winter)', ascending=False).round(4)
df_season_phase_test.to_csv(DATA_DIR + 'seasonal_phase_test.csv', index=False)
df_season_phase_test

## Hospital density relationships

In [ ]:
rows = []
for i, region in enumerate(regions):
    density = hosp_df.loc[hosp_df['Region'] == region, 'EDHospitalsPer10k'].iat[0]
    b = raw_df[f'beta[{i+1}]'].values
    g = raw_df[f'gamma[{i+1}]'].values
    alpha = raw_df[f'alpha[{i+1}]'].values
    phase_rad = np.arctan2(g, b)
    rows.append({
        'Region': region,
        'EDHospitalsPer10k': density,
        'Alpha Mean': alpha.mean(),
        'Beta Mean': b.mean(),
        'Gamma Mean': g.mean(),
        'Amplitude Mean': ampl[region].mean(),
        'Amplitude 2.5%': np.quantile(ampl[region], 0.025),
        'Amplitude 97.5%': np.quantile(ampl[region], 0.975),
        'P(A > 0.5)': (ampl[region] > epsilon).mean(),
        'Mean Peak Week': ((52 / (2 * np.pi)) * phase_rad.mean()) % 52,
    })

df_density_joined = pd.DataFrame(rows).sort_values('EDHospitalsPer10k', ascending=False).round(4)
df_density_joined.to_csv(DATA_DIR + 'hospital_density_joined_summary.csv', index=False)

corr_rows = []
for metric in ['Beta Mean', 'Gamma Mean', 'Amplitude Mean', 'P(A > 0.5)']:
    corr_rows.append({
        'Metric': metric,
        'Correlation with density': df_density_joined['EDHospitalsPer10k'].corr(df_density_joined[metric])
    })

df_density_corr = pd.DataFrame(corr_rows).round(4)
df_density_corr.to_csv(DATA_DIR + 'hospital_density_correlations.csv', index=False)
df_density_joined

In [ ]:
plot_labeled_scatter(
    df_density_joined,
    'EDHospitalsPer10k',
    'Amplitude Mean',
    'Region',
    PLOT_DIR + 'density_vs_amplitude.png',
    'Hospital Density vs Seasonal Amplitude',
    xlabel='ED hospitals per 10k',
    ylabel='Posterior mean amplitude'
)

plot_labeled_scatter(
    df_density_joined,
    'EDHospitalsPer10k',
    'P(A > 0.5)',
    'Region',
    PLOT_DIR + 'density_vs_amplitude_probability.png',
    'Hospital Density vs P(A > 0.5)',
    xlabel='ED hospitals per 10k',
    ylabel='Posterior exceedance probability'
)

plot_labeled_scatter(
    df_density_joined,
    'EDHospitalsPer10k',
    'Beta Mean',
    'Region',
    PLOT_DIR + 'density_vs_beta.png',
    'Hospital Density vs Mean Cosine Coefficient',
    xlabel='ED hospitals per 10k',
    ylabel='Posterior mean beta'
)

plot_labeled_scatter(
    df_density_joined,
    'EDHospitalsPer10k',
    'Gamma Mean',
    'Region',
    PLOT_DIR + 'density_vs_gamma.png',
    'Hospital Density vs Mean Sine Coefficient',
    xlabel='ED hospitals per 10k',
    ylabel='Posterior mean gamma'
)

df_density_corr

## Intercepts

V6 does not partially pool or regress the intercept on hospital density. The intercept summaries are retained here so V6 remains directly comparable with the earlier region-level models.

In [ ]:
alpha_samples = {regions[i]: raw_df[f'alpha[{i+1}]'].values for i in range(n_region)}

alpha_rows = []
for region in regions:
    vals = alpha_samples[region]
    alpha_rows.append({
        'Region': region,
        'Mean': vals.mean(),
        'SD': vals.std(),
        '2.5%': np.quantile(vals, 0.025),
        '97.5%': np.quantile(vals, 0.975),
    })

df_alpha_overall = pd.DataFrame(alpha_rows).sort_values('Mean', ascending=False).round(4)
df_alpha_overall.to_csv(DATA_DIR + 'alpha_overall.csv', index=False)

df_alpha_pw = pairwise_test(alpha_samples, regions)
df_alpha_pw.to_csv(DATA_DIR + 'alpha_pairwise.csv', index=False)
df_alpha_overall

In [ ]:
pairwise_heatmap(
    df_alpha_pw,
    'P(diff > 0)',
    r'Alpha ($\alpha_i$): P(Region A > Region B)',
    PLOT_DIR + 'alpha_pairwise.png'
)
df_alpha_pw